In [ ]:
import torch
torch.backends.cudnn.benchmark = True
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Using device:", device)

# Initialise Stem class
class Stem(nn.Module):

  def __init__(self):
    # Create a convolutional layer
    super(Stem, self).__init__()
    self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1)

    # Create max pooling layer
    self.pool = nn.MaxPool2d(kernel_size=3, stride=1, padding=0)



# Define forward pass
  def forward(self, x):

    x = self.conv1(x) # Convolutional layer to extract features
    x = F.relu(x) # Apply non-liearity
    x = self.pool(x) # Down-sample using max pooling

    return x

# Define Expert Branch
class ExpertBranch(nn.Module):

  # Expert Branch Constructor

  # in_channels: Number of input chanels from the previous layer
  # reduction_ratio: Used to shrink the number of channels in the intermediate layer for efficiency
  # num_kernels: How many different convolutional branches/kernels we're learning to to weight

  def __init__(self, in_channels, reduction_ratio=2, num_kernels=5):

    super(ExpertBranch, self).__init__() # Calls the parent nn.Module constructor

    reduced_channels = in_channels // reduction_ratio # Reduces number of channels in the middle layer

    self.pool = nn.AdaptiveAvgPool2d(1) # Applies global average pooling to each feature map

    self.fc1 = nn.Linear(in_channels, reduced_channels) # First Fully Connected layer
    self.fc2 = nn.Linear(reduced_channels, num_kernels) # Second Fulle Connected layer


# Define the forward pass (how data flows through layers)
  def forward(self, x):

    batch_size , channels, _, _ = x.size() # Reads input tensor shape

    x = self.pool(x).view(batch_size, channels) # Apple global avg pooling over spatial dimensions

    x = F.relu(self.fc1(x)) # Transform input into a compact latent representation (non-linear activation)
    x = F.softmax(self.fc2(x), dim=1) # Gives probabiity distribution over the expert branches

    return x # Return final softmaxed vector of attention weights


# Define convolutional branch that contains multiple parallel convolutional layers
class ConvBranch(nn.Module):

  def __init__(self, in_channels, out_channels, kernel_size=3, num_kernels=5):

    super(ConvBranch, self).__init__() # Initialises parent class so PyTorch can manage parameters

    # self.convs is a list of convolutional layers
    self.convs = nn.ModuleList([
        nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1), # padding ensures output spatial size is preserved when using kernel_size
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
        
        for _ in range(num_kernels)
    ])

#
  def forward(self, x, weights):
    out = 0 # initialises final output as zero

    # Loop over each convolution layer and its index
    for i, conv in enumerate(self.convs):
      weighted_output = weights[:, i].view(-1, 1, 1, 1) * conv(x)
      out += weighted_output

    return out

# Define new block dynamically combining multiple convolutional outputs based on the learned attention
class DynamicBlock(nn.Module):

  # Initialise block with input, output channels and number of kernels
  def __init__(self, in_channels, out_channels, num_kernels=4):

    super(DynamicBlock, self).__init__() # Calls parent constructor

    # Initialise expert branch
    self.expert = ExpertBranch(in_channels, num_kernels=num_kernels)

    # Initialise convolutional branch
    self.conv_branch = ConvBranch(in_channels, out_channels, num_kernels=num_kernels)

# Define how the input flows through dynamic block
  def forward(self, x):

    # Passes input through expert branch, gets attention weights for each
    weights = self.expert(x)

    # Apply conv branches using learned weights to combine their outputs
    x = self.conv_branch(x, weights)

    return x # Return combined output

# Define classifier module for final imagge classification
class Classifier(nn.Module):

  # Initialise classifier
  def __init__(self, in_features, num_classes):

    super(Classifier, self).__init__() # Fully connected layer that maps extracted features

    self.fc = nn.Linear(in_features, num_classes)

# Define Input flow
  def forward(self, x):

    x = F.adaptive_avg_pool2d(x, 1) # Apply adaptive average pooling to reduce each feature map to a single value (global avg pooling)
    x = x.view(x.size(0), -1) # Flatten pooled feature map to single value
    x = self.fc(x) # Apply fully connected layer to get final class scores

    return x # Return raw logits

# Define full dynamic convolutional neural network model
class DynamicConvNet(nn.Module):

  # Initialise layers of the model
  def __init__(self, num_blocks=4, num_kernels=5, num_classes=10):

    super(DynamicConvNet, self).__init__() # Inherit from nn.Module

    print("Initialising DynamicConvNet model...")

    self.stem = Stem() # Initialise feature extraction layer

    blocks = [] # List to store all dynamic blocks

    in_channels = 64 # Number of channels output from Stem layer

    # Create sequence of dynamic blocks
    for _ in range(num_blocks):
      # Each block takes in parameters
      blocks.append(DynamicBlock(in_channels, in_channels, num_kernels=num_kernels))

    self.backbone = nn.Sequential(*blocks) # Combine list of dynamic blocks into one module

    self.classifier = Classifier(in_channels, num_classes) # Final classification layer to predict the class of the input image

    self.dropout = nn.Dropout(p=0.3)
  # Define how data flows through the full network
  def forward(self, x):

    x = self.stem(x) # Visual features: edges, shapes, textures

    x = self.backbone(x) # Pass through dynamic convolutional layers

    x = self.dropout(x)

    x = self.classifier(x) # VConvert final feature maps to logits

    return x # Return logits


# Datasets and Training Components

# Define transformaitions and datasets
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(), # Converts image to PyTorch tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalises tensor images (RGB)
]) # Pipeline of transformations applied to every image in dataset

# CIFAR-10 dataset
# Downloads CIFAR-10 training dataset, applies transformation pipeline to each loaded image
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
#Loads data in mini-batches of 32
trainloader = DataLoader(trainset, batch_size=128, shuffle=True)

# Specifies loading the test set
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
# Manages how the CIFAR-10 test dataset is loaded during evaluation phase
testloader = DataLoader(testset, batch_size=128, shuffle=False)

# Initialises model class with set parameters
model = DynamicConvNet(num_blocks=4, num_kernels=5, num_classes=10).to(device)

# Training The Model
num_epochs = 150 # Number of times the entire training dataset will be passed through the model

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.2) # Loss function (cross-entropy loss)
optimiser = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=5e-4) # Adapts learning rate
scheduler = CosineAnnealingLR(optimiser, T_max=num_epochs, eta_min=1e-6)



train_losses = []
test_accuracies = []
train_accuracies = []

# Loop running the epochs to train the model
for epoch in range(num_epochs):

  model.train() # Puts model in training mode
  running_loss = 0.0 # Initialises variable to track cumulative loss for the current epoch
  correct = 0 # Initialises variable to count number of correct predictions for the current epoch
  total = 0 # Initalises variable to count total number of examples processed furing the current epoch

# Loop through training dataset in mini-batches (trainloader), inputs are the input images, labels are corresponding ground truth labels
  for inputs, labels in trainloader:

    inputs, labels = inputs.to(device), labels.to(device)
    optimiser.zero_grad() # Clears gradients of all optimised tensors

    outputs = model(inputs.to(device)) # Passes input images through model to get logits

    loss = criterion(outputs, labels.to(device)) # Calculates loss - compares predicted outputs with true labels using CrossEntropyLoss

    loss.backward() # Computes gradients of loss with respect to models parameters (backpropagation)

    optimiser.step() # Updates models parameters based on the calculated gradients and optimisation algorithm (Adam)

    running_loss += loss.item() # Adds current batch's loss to running total

    _, predicted = torch.max(outputs, 1) # Gets predicted class labels - selects class with highest predicted score from models logits
    total += labels.size(0) # Updates total number of samples processed
    correct += (predicted == labels).sum().item() # Counts predictions matching true labels

    current_lr = optimiser.param_groups[0]['lr']

# Output current epochs progress, includes average loss and accuracy.
  scheduler.step()
  print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(trainloader):.4f}, Accuracy: {100 * correct / total:.2f}%, Learning Rate: {current_lr:.6f}')

  train_accuracy = 100 * correct / total
  train_accuracies.append(train_accuracy)
  train_losses.append(running_loss / len(trainloader))
# Model Evaluation

  model.eval() # Puts model in evaluation mode
  correct = 0 # Initialises variable to track number of correctly predicted labels
  total = 0 # Initialises variable to count total number of labels in test dataset

  # Disables gradient computation
  with torch.no_grad():

    # Loop over test data
    for inputs, labels in testloader:

      inputs, labels = inputs.to(device), labels.to(device)

      outputs = model(inputs) # Feeds inputs through model to get logits
      _, predicted = torch.max(outputs, 1) # Computes predicted class for each input - selects class with highest logit value

      total += labels.size(0) # Adds batch size to total count
      correct += (predicted == labels).sum().item() # Compares predicted labels with true labels

    test_accuracy = 100 * correct / total
    test_accuracies.append(test_accuracy)
  # Calculates and prints accuracy of model - divides correct predictions by total number of labels
print(f'Test Accuracy: {100 * correct / total:.2f}%')
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label='Training Loss', color='blue')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_accuracies, label='Train Accuracy', color='green')
plt.plot(test_accuracies, label='Test Accuracy', color='red')
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curves")
plt.legend()

plt.tight_layout()
plt.show()
